<a href="https://colab.research.google.com/github/bogdanbabych/experiments_NLTK/blob/main/part2_word_embeddings_v09_HIS_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Word Vectors

Presentations [slides](intro-word-vectors-DAAD-2021-v22.pdf); Original [slides](word-vectors.pdf)

Word vectors (also known as 'word embeddings') are one of the most popular kinds of AI models. They are extremely useful in many domains. In essence, a word vector is a set of numbers that attempt to capture the meaning of a word. In typical implementations, each word is represented by a set of 200-300 numbers. In linear algebra, a one-dimensional array of numbers is known as a 'vector', hence these sets of numbers representing words' meanings are known as 'word vectors'.

Using neural networks, we can expose the computer to a large amount of text, and allow it to learn an appropriate set of numbers for each word it encounters. In this notebook, we will learn about the most famous of all word vector algorithms, `word2vec`, which was first described by Tomas Mikolov and his team in 2013:

* Tomas Mikolov, Ilya Sutskever, and others, ‘Distributed Representations of Words and Phrases and Their Compositionality’, in Advances in Neural Information Processing Systems 26, ed. by C. J. C. Burges and others (Curran Associates, Inc., 2013), pp. 3111–19 <http://papers.nips.cc/paper/5021-distributed-representations-of-words-and-phrases-and-their-compositionality.pdf>
* Tomas Mikolov, Kai Chen, and others, ‘Efficient Estimation of Word Representations in Vector Space’, ArXiv:1301.3781 Cs, 2013 <http://arxiv.org/abs/1301.3781>.

In fact, `word2vec` is not a single algorithm, but rather a family of similar algorithms. In this session we will consider just the most famous `word2vec` algorithm, namely the `skip-gram model` trained using `negative sampling`.

## Applications of Word Vectors

Word vectors allow the computer to 'understand' language far more effectively. Rather than seeing each word as simply an arbitrarily different object, a computer using word vectors can analyse each word as a point in 200- or 300-dimenstional space. Words that are similar in meaning will have similar word vectors. And as we will see, the spaces between the word vectors are also significant: the words are arranged in patterns that represent their relationships to one another.

Accordingly, most AI systems that process language now include a word vector layer as part of their architecure. When the system encounters some text (e.g. when you speak to Siri or Alexa), your words are converted into word vectors, *and then* the computer examines what the text says and determines how it should respond.

In the Humanities, word vectors have become a popular modelling tool, because they allow researchers to perform sophisticated analysis on large corpora of text. Some examples include:

* [The Women Writers Vector Toolkit](https://wwp.northeastern.edu/lab/wwvt/index.html)
* William L. Hamilton, Jure Leskovec, and Dan Jurafsky, ‘Diachronic Word Embeddings Reveal Statistical Laws of Semantic Change’, ArXiv:1605.09096 [Cs], 2018 <http://arxiv.org/abs/1605.09096>.
* Ryan Heuser, 'Semantic Networks' <https://ryanheuser.org/word-vectors-4/>

## Training a `word2vec` model in Gensim


It is very easy to train a `word2vec` model in Gensim, which includes Mikolov's original `word2vec` code in its codebase.

### Step 0: Downloading texts
#### 0.1: load an existing text collection...


In [2]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 66.6 MB/s eta 0:00:00


In [3]:
from gensim.models import Word2Vec # The word2vec model class
import gensim.downloader as api # Allows us to download some free training data

# api.info()

In [4]:
corpus = api.load('text8')
word_vectors = api.load('glove-wiki-gigaword-300')
data = [d for d in corpus]

[==================================================] 100.0% 31.6/31.6MB downloaded
[==================================================] 100.0% 376.1/376.1MB downloaded


In [ ]:
#

text8 is first 100MB from English wikipedia, 17 million words

The file is downloadable from
http://mattmahoney.net/dc/text8.zip


In [5]:
# Examine the corpus to see what is there
api.info("text8")

{'num_records': 1701,
 'record_format': 'list of str (tokens)',
 'file_size': 33182058,
 'reader_code': 'https://github.com/RaRe-Technologies/gensim-data/releases/download/text8/__init__.py',
 'license': 'not found',
 'description': 'First 100,000,000 bytes of plain text from Wikipedia. Used for testing purposes; see wiki-english-* for proper full Wikipedia datasets.',
 'checksum': '68799af40b6bda07dfa47a32612e5364',
 'file_name': 'text8.gz',
 'read_more': ['http://mattmahoney.net/dc/textdata.html'],
 'parts': 1}

In [8]:
type(corpus)

text8.Dataset

In [ ]:
# data = [d for d in corpus]

In [7]:
len(data)

1701

In [9]:
len(data[0])

10000

In [10]:
len(data[1])

10000

In [11]:
len(data[2])

10000

In [12]:
print(data[0][:100])

['anarchism', 'originated', 'as', 'a', 'term', 'of', 'abuse', 'first', 'used', 'against', 'early', 'working', 'class', 'radicals', 'including', 'the', 'diggers', 'of', 'the', 'english', 'revolution', 'and', 'the', 'sans', 'culottes', 'of', 'the', 'french', 'revolution', 'whilst', 'the', 'term', 'is', 'still', 'used', 'in', 'a', 'pejorative', 'way', 'to', 'describe', 'any', 'act', 'that', 'used', 'violent', 'means', 'to', 'destroy', 'the', 'organization', 'of', 'society', 'it', 'has', 'also', 'been', 'taken', 'up', 'as', 'a', 'positive', 'label', 'by', 'self', 'defined', 'anarchists', 'the', 'word', 'anarchism', 'is', 'derived', 'from', 'the', 'greek', 'without', 'archons', 'ruler', 'chief', 'king', 'anarchism', 'as', 'a', 'political', 'philosophy', 'is', 'the', 'belief', 'that', 'rulers', 'are', 'unnecessary', 'and', 'should', 'be', 'abolished', 'although', 'there', 'are', 'differing']


In [13]:
print(data[1][:100])

['reciprocity', 'qualitative', 'impairments', 'in', 'communication', 'as', 'manifested', 'by', 'at', 'least', 'one', 'of', 'the', 'following', 'delay', 'in', 'or', 'total', 'lack', 'of', 'the', 'development', 'of', 'spoken', 'language', 'not', 'accompanied', 'by', 'an', 'attempt', 'to', 'compensate', 'through', 'alternative', 'modes', 'of', 'communication', 'such', 'as', 'gesture', 'or', 'mime', 'in', 'individuals', 'with', 'adequate', 'speech', 'marked', 'impairment', 'in', 'the', 'ability', 'to', 'initiate', 'or', 'sustain', 'a', 'conversation', 'with', 'others', 'stereotyped', 'and', 'repetitive', 'use', 'of', 'language', 'or', 'idiosyncratic', 'language', 'lack', 'of', 'varied', 'spontaneous', 'make', 'believe', 'play', 'or', 'social', 'imitative', 'play', 'appropriate', 'to', 'developmental', 'level', 'restricted', 'repetitive', 'and', 'stereotyped', 'patterns', 'of', 'behavior', 'interests', 'and', 'activities', 'as', 'manifested', 'by', 'at', 'least', 'one']


In [16]:
print(data[-2][:100])

['four', 'five', 'four', 'eight', 'four', 'nine', 'five', 'zero', 'five', 'two', 'five', 'four', 'five', 'six', 'six', 'zero', 'six', 'three', 'if', 'n', 'is', 'prime', 'then', 'n', 'one', 'but', 'the', 'converse', 'is', 'not', 'true', 'the', 'first', 'non', 'prime', 'n', 'for', 'which', 'n', 'one', 'is', 'three', 'zero', 'two', 'three', 'five', 'the', 'first', 'such', 'numbers', 'with', 'three', 'distinct', 'prime', 'factors', 'sphenic', 'numbers', 'are', 'oeis', 'a', 'zero', 'zero', 'seven', 'three', 'zero', 'four', 'three', 'zero', 'four', 'two', 'six', 'six', 'seven', 'zero', 'seven', 'eight', 'one', 'zero', 'two', 'one', 'zero', 'five', 'one', 'one', 'zero', 'one', 'one', 'four', 'one', 'three', 'zero', 'one', 'three', 'eight', 'one', 'five', 'four', 'one', 'six', 'five']


#### Step 0.2: load own text collection to be used in training ... (will be added later)...


### Step 1: Set hyperparameters and instantiate model

In [17]:
my_vector_size = 100 # Dimensionality of the word vectors
window = 5 # How many words either side? (5 = 5 context words either side, i.e. 10 context words in total)
use_skip_gram = 1 # If you set this to 0, then it will create a 'continuous bag of words' model instead
use_softmax = 0 # If you set this to 1, then hierarchical softmax will be used instead of negative sampling
negative_samples = 5 # How many incorrect answers to generate per correct answer when negative sampling

model = Word2Vec(
    vector_size=my_vector_size,
    window=window,
    sg=use_skip_gram,
    hs=use_softmax,
    negative=negative_samples
)

### Step 2: Fit model to corpus

In [18]:
# build a model
model.build_vocab(corpus)

In [19]:
# Train the model on the corpus
# model.train(sentences=corpus, epochs=5, total_examples=model.corpus_count)
# runs approximately 10 minutes with 5 epochs...
model.train(corpus, epochs=5, total_examples=model.corpus_count)

# saving the model, zip can be downloaded...
model.save("word2vec5e.model")
!zip word2vec5e.model.zip word2vec5e.model
!rm word2vec5e.model

  adding: word2vec5e.model (deflated 10%)


In [20]:
!rm word2vec5e.model.zip

In [21]:
!wget https://heibox.uni-heidelberg.de/f/bf6ec8bbff5b43e4b646/?dl=1
!mv index.html?dl=1 word2vec5e.model.zip
!unzip word2vec5e.model.zip

--2026-06-28 12:06:11--  https://heibox.uni-heidelberg.de/f/bf6ec8bbff5b43e4b646/?dl=1
Resolving heibox.uni-heidelberg.de (heibox.uni-heidelberg.de)... 129.206.7.113
Connecting to heibox.uni-heidelberg.de (heibox.uni-heidelberg.de)|129.206.7.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://heibox.uni-heidelberg.de/seafhttp/files/7e8e611b-7edc-4b56-8509-205cdd4d41d6/word2vec5e.model.zip [following]
--2026-06-28 12:06:11--  https://heibox.uni-heidelberg.de/seafhttp/files/7e8e611b-7edc-4b56-8509-205cdd4d41d6/word2vec5e.model.zip
Reusing existing connection to heibox.uni-heidelberg.de:443.
HTTP request sent, awaiting response... 200 OK
Length: 53495855 (51M) [application/zip]
Saving to: ‘index.html?dl=1’

index.html?dl=1     100%[===================>]  51.02M  20.2MB/s    in 2.5s    

2026-06-28 12:06:14 (20.2 MB/s) - ‘index.html?dl=1’ saved [53495855/53495855]

Archive:  word2vec5e.model.zip
  inflating: word2vec5e.model        


In [ ]:
# modelOwn.train(sentences=corpusOwn, epochs=5, total_examples=modelOwn.corpus_count)

In [ ]:
## !rm word2vec.model

### Step 2: alternative: downloading a pre-trained model

In [ ]:
## Downloading and connecting pre-trained model
!wget https://heibox.uni-heidelberg.de/f/b7625cd746534aacbefe/?dl=1
!mv index.html?dl=1 word2vec.model

--2024-06-09 14:38:38--  https://heibox.uni-heidelberg.de/f/b7625cd746534aacbefe/?dl=1
Resolving heibox.uni-heidelberg.de (heibox.uni-heidelberg.de)... 129.206.7.113
Connecting to heibox.uni-heidelberg.de (heibox.uni-heidelberg.de)|129.206.7.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://heibox.uni-heidelberg.de/seafhttp/files/2dc0b8fa-7013-495a-bf24-9bd188c1ce72/word2vec.model [following]
--2024-06-09 14:38:38--  https://heibox.uni-heidelberg.de/seafhttp/files/2dc0b8fa-7013-495a-bf24-9bd188c1ce72/word2vec.model
Reusing existing connection to heibox.uni-heidelberg.de:443.
HTTP request sent, awaiting response... 200 OK
Length: 59217897 (56M) [application/octet-stream]
Saving to: ‘index.html?dl=1’

index.html?dl=1     100%[===================>]  56.47M  17.0MB/s    in 3.5s    

2024-06-09 14:38:42 (16.2 MB/s) - ‘index.html?dl=1’ saved [59217897/59217897]



In [22]:
model = Word2Vec.load("word2vec5e.model")

In [ ]:
model = Word2Vec.load("word2vec.model")
# model2 = Word2Vec.load("word2vec.model")

### Step 3: Extract word vectors from model

The fully trained model includes all of the weights used to predict the context words for each input word. If you are not planning on training the model further, these weights can be discarded, and you can just keep the weights for the word vectors.

In [23]:
word_vectors2 = model.wv
del model # Delete the whole model to free up the computer's RAM

In [24]:
word_vectors2.save("word2vec.wordvectors5e")
!zip word2vec.wordvectors5e.zip word2vec.wordvectors5e
!rm word2vec.wordvectors5e

  adding: word2vec.wordvectors5e (deflated 12%)


In [28]:
!unzip word2vec.wordvectors5e.zip

Archive:  word2vec.wordvectors5e.zip
  inflating: word2vec.wordvectors5e  


In [ ]:
## !rm word2vec.wordvectors

In [88]:
word_vectors.save("word2vec.wordvectors")
# !zip word2vec.wordvectors.zip word2vec.wordvectors
# !rm word2vec.wordvectors

In [90]:
del word_vectors

In [ ]:
word_vectors = Word2Vec.load("word2vec.wordvectors")

### Step 3 alternative: download and connect the pre-trained word vectors

In [ ]:
!wget https://heibox.uni-heidelberg.de/f/e7725b1afab9456c8e16/?dl=1
!mv index.html?dl=1 word2vec.wordvectors

In [ ]:
from gensim.models import KeyedVectors

In [29]:
word_vectors3 = KeyedVectors.load("word2vec.wordvectors5e", mmap='r')

In [ ]:

word_vectors3 = KeyedVectors.load("word2vec.wordvectors", mmap='r')

### Step 3 alternative-2: downloading a pre-trained Gigaword model, 300 dimensions... (~5 min)

In [ ]:
# word_vectors = api.load('glove-wiki-gigaword-300')

[==================================================] 100.0% 376.1/376.1MB downloaded


### Step 4: exploring word vectors

In [32]:
vector = word_vectors3['computer']
vector

array([ 0.25097626,  0.0421246 ,  0.09677728, -0.29220584, -0.2702313 ,
        0.05108112,  0.2583029 ,  0.4209945 , -0.4133952 , -0.6046314 ,
        0.6810764 ,  0.23518127, -0.08793572, -0.14458553, -0.10153013,
       -0.25877467, -0.08860687,  0.22598608, -0.12342963, -0.6107972 ,
        0.22940446,  0.06515529,  0.09304131, -0.36797452,  0.11043684,
        0.40758005,  0.05187347, -0.29520005, -0.5446818 , -0.16412328,
       -0.00559328,  0.11292659,  0.92568225, -0.58871126,  0.00174291,
        0.26249707,  0.37171453, -0.04846693,  0.08307406, -0.18562861,
       -0.23446865, -0.57738936,  0.76738864,  0.55871767,  0.282225  ,
       -0.37604094, -0.07775856,  0.5470369 , -0.14910425,  0.09307888,
        0.07012475, -0.11130489, -0.08199138, -0.56308025, -0.4849242 ,
       -0.05054295,  0.21772623, -0.60420305,  0.27237868,  0.19701481,
        0.43979117, -0.4106916 ,  0.03609947,  0.0145739 , -0.12259277,
        0.70135826,  0.08454874,  0.00822722, -0.4179959 ,  0.13

In [33]:
print(vector)

[ 0.25097626  0.0421246   0.09677728 -0.29220584 -0.2702313   0.05108112
  0.2583029   0.4209945  -0.4133952  -0.6046314   0.6810764   0.23518127
 -0.08793572 -0.14458553 -0.10153013 -0.25877467 -0.08860687  0.22598608
 -0.12342963 -0.6107972   0.22940446  0.06515529  0.09304131 -0.36797452
  0.11043684  0.40758005  0.05187347 -0.29520005 -0.5446818  -0.16412328
 -0.00559328  0.11292659  0.92568225 -0.58871126  0.00174291  0.26249707
  0.37171453 -0.04846693  0.08307406 -0.18562861 -0.23446865 -0.57738936
  0.76738864  0.55871767  0.282225   -0.37604094 -0.07775856  0.5470369
 -0.14910425  0.09307888  0.07012475 -0.11130489 -0.08199138 -0.56308025
 -0.4849242  -0.05054295  0.21772623 -0.60420305  0.27237868  0.19701481
  0.43979117 -0.4106916   0.03609947  0.0145739  -0.12259277  0.70135826
  0.08454874  0.00822722 -0.4179959   0.1303945   0.20154981  0.68534875
  0.23772904  0.3759207   0.35068634 -0.00834582  0.28801385  0.06428442
 -0.14504628 -0.19663239 -0.7203334  -0.44208804  0.

### Step 4: Have a play with the model

There are several ways you can use word vectors. One of the most famous is to use them to compute analogies. The formula is:

<center><em>x</em> is to <em>small</em> as <em>biggest</em> is to <em>big</em></center>

$$x - vector('small') = vector('biggest') - vector('big')$$

$$\therefore x = vector('small') + vector('biggest') - vector('big')$$

In [109]:
# See the word vector for a particular word
vector = word_vectors['banana']
print(vector)

[ 4.2141e-01  2.0467e-02  1.2666e-01  3.9762e-01 -1.1016e-01 -3.5956e-02
 -4.7214e-01 -1.3916e-01  5.6812e-01 -3.4969e-01 -9.3232e-02 -1.7035e-01
 -3.8677e-01 -1.6811e-01 -1.0157e-01 -2.6612e-01  4.8094e-02 -4.6771e-01
 -6.0725e-01  4.0952e-01  3.1771e-01  5.0098e-01  6.6368e-01 -1.1827e-01
 -7.4267e-01 -1.0472e-01 -6.4353e-01 -4.4023e-01 -3.9101e-01  3.5694e-01
 -9.3489e-01  4.8317e-01  1.5223e-01  7.9339e-02 -2.5111e-01  3.9968e-01
 -1.7982e-01 -2.8874e-01 -1.0891e-01  3.8821e-01 -2.3147e-01 -5.0337e-01
 -2.5231e-01 -2.2184e-02 -2.7874e-01 -2.4193e-01  5.7466e-02 -5.3955e-01
 -3.4875e-02 -4.0482e-01 -3.8067e-02 -4.2337e-01  4.2861e-01  3.5166e-01
 -1.8165e-01 -3.1131e-01 -5.3276e-01 -5.0954e-02  6.6779e-01 -4.0077e-01
  2.1403e-01 -2.9861e-01 -3.6637e-01  2.8489e-01 -3.7663e-01  5.9604e-02
 -3.1795e-01  2.5463e-01 -2.2185e-01  2.3032e-01 -1.2302e-01  2.4175e-01
 -1.0706e-01 -8.6599e-02 -3.7363e-02 -1.0403e-01  2.4492e-01 -8.4063e-01
 -1.5350e-01 -1.9363e-01 -1.8541e-02  1.0938e-01 -2

In [97]:
# See which words are closest to a given word in the vector space
similar_words = word_vectors.most_similar('university', topn=25)
print('\n'.join([str(tup) for tup in similar_words]))

('professor', 0.7463629841804504)
('graduate', 0.7044455409049988)
('college', 0.6996859312057495)
('harvard', 0.6874451041221619)
('faculty', 0.6825277209281921)
('yale', 0.6606155037879944)
('universities', 0.6385825872421265)
('campus', 0.6320205330848694)
('doctorate', 0.6097204685211182)
('stanford', 0.600467324256897)
('student', 0.5970513224601746)
('graduated', 0.5920378565788269)
('studies', 0.5913025140762329)
('science', 0.5849512815475464)
('cornell', 0.5826446413993835)
('princeton', 0.5791559219360352)
('school', 0.5767619609832764)
('undergraduate', 0.5731509327888489)
('institute', 0.5731453895568848)
('students', 0.5693991780281067)
('graduating', 0.5659559965133667)
('studied', 0.5610754489898682)
('georgetown', 0.5545408129692078)
('degree', 0.5524526834487915)
('lecturer', 0.5508397221565247)


In [36]:
# See which words are closest to a given word in the vector space
similar_words = word_vectors.most_similar('red', topn=25)
print('\n'.join([str(tup) for tup in similar_words]))

('yellow', 0.682881772518158)
('blue', 0.6736692786216736)
('pink', 0.5893942713737488)
('green', 0.5799195766448975)
('white', 0.5733830332756042)
('purple', 0.5530810356140137)
('black', 0.5441405177116394)
('colored', 0.5216451287269592)
('sox', 0.5209404826164246)
('bright', 0.5119451284408569)
('dark', 0.48857060074806213)
('orange', 0.48808690905570984)
('colors', 0.48122531175613403)
('wearing', 0.46445152163505554)
('wore', 0.4515107274055481)
('colours', 0.45110228657722473)
('wings', 0.4426623582839966)
('uniforms', 0.44212859869003296)
('flags', 0.43636879324913025)
('color', 0.4353710412979126)
('flag', 0.4304704964160919)
('cross', 0.43032655119895935)
('gray', 0.42928779125213623)
('grey', 0.42531874775886536)
('crimson', 0.4218621551990509)


In [37]:
similar_words = word_vectors.most_similar('color', topn=25)
print('\n'.join([str(tup) for tup in similar_words]))

('colour', 0.8112421035766602)
('colors', 0.7267135381698608)
('colored', 0.6027425527572632)
('colours', 0.5306311249732971)
('shades', 0.5302167534828186)
('palette', 0.5084972381591797)
('bright', 0.5052652955055237)
('yellow', 0.5037733316421509)
('texture', 0.5014382600784302)
('purple', 0.4947432577610016)
('images', 0.49079400300979614)
('photo', 0.4904259443283081)
('background', 0.48688817024230957)
('pink', 0.4852536618709564)
('prints', 0.4836598038673401)
('coloured', 0.47849053144454956)
('screen', 0.4764823913574219)
('hues', 0.4736193120479584)
('print', 0.4711713194847107)
('blue', 0.47014671564102173)
('coloring', 0.46921345591545105)
('image', 0.46893927454948425)
('paint', 0.46414002776145935)
('feature', 0.45886921882629395)
('dark', 0.45610302686691284)


In [38]:
similar_words = word_vectors.most_similar('colour', topn=25)
print('\n'.join([str(tup) for tup in similar_words]))

('color', 0.8112421035766602)
('colours', 0.6802125573158264)
('colors', 0.6598658561706543)
('coloured', 0.6242401599884033)
('coloration', 0.5722349882125854)
('reddish', 0.5671398043632507)
('yellow', 0.5559722185134888)
('shades', 0.5441486835479736)
('yellowish', 0.5363735556602478)
('brownish', 0.5225512981414795)
('darker', 0.5125751495361328)
('colored', 0.5070598721504211)
('purple', 0.5039236545562744)
('bluish', 0.5011844038963318)
('greenish', 0.49972444772720337)
('palette', 0.49671489000320435)
('hues', 0.49636197090148926)
('markings', 0.4936930537223816)
('distinctive', 0.4924719035625458)
('texture', 0.48974609375)
('monochrome', 0.482740581035614)
('reddish-brown', 0.47975584864616394)
('pale', 0.4778788089752197)
('pinkish', 0.47278302907943726)
('colouration', 0.47110632061958313)


In [98]:
# See which words are closest to a given word in the vector space
similar_words = word_vectors.most_similar('brussels', topn=25)
print('\n'.join([str(tup) for tup in similar_words]))

('eu', 0.5924946665763855)
('paris', 0.5912899374961853)
('belgium', 0.5622056126594543)
('prohertrib', 0.5407592058181763)
('vienna', 0.5333185195922852)
('amsterdam', 0.5311341881752014)
('european', 0.5278900265693665)
('bonn', 0.522121787071228)
('lisbon', 0.5173484683036804)
('bucharest', 0.5105912089347839)
('copenhagen', 0.509868323802948)
('budapest', 0.509274423122406)
('berlin', 0.504755973815918)
('ankara', 0.4947936534881592)
('strasbourg', 0.49196168780326843)
('nato', 0.4852606952190399)
('moscow', 0.485049307346344)
('warsaw', 0.48083314299583435)
('luxembourg', 0.4734221398830414)
('meeting', 0.4726783037185669)
('prague', 0.46613407135009766)
('belgrade', 0.46501246094703674)
('london', 0.46029549837112427)
('rome', 0.455429345369339)
('geneva', 0.4536011517047882)


In [99]:
# See which words are closest to a given word in the vector space
similar_words = word_vectors.most_similar('heidelberg', topn=25)
print('\n'.join([str(tup) for tup in similar_words]))

('göttingen', 0.6687851548194885)
('tübingen', 0.6459972262382507)
('mannheim', 0.6052619814872742)
('würzburg', 0.6046574711799622)
('leipzig', 0.5680612325668335)
('giessen', 0.5450029969215393)
('freiburg', 0.5425687432289124)
('breslau', 0.5277659893035889)
('erlangen', 0.5239335894584656)
('karlsruhe', 0.5130500197410583)
('darmstadt', 0.5068687200546265)
('augsburg', 0.49566495418548584)
('münster', 0.49557027220726013)
('bonn', 0.48971569538116455)
('königsberg', 0.48326921463012695)
('cologne', 0.47586944699287415)
('universität', 0.4744783341884613)
('düsseldorf', 0.46852555871009827)
('stuttgart', 0.46200859546661377)
('kassel', 0.4564279615879059)
('berlin', 0.4551321864128113)
('marburg', 0.4550616443157196)
('baden-württemberg', 0.4497103989124298)
('magdeburg', 0.44885918498039246)
('braunschweig', 0.4457882344722748)


In [100]:
# See which words are closest to a given word in the vector space
similar_words = word_vectors.most_similar('international', topn=25)
print('\n'.join([str(tup) for tup in similar_words]))

('global', 0.565397322177887)
('world', 0.5379492044448853)
('nations', 0.49859651923179626)
('based', 0.49091580510139465)
('organizations', 0.4790637493133545)
('organization', 0.47753334045410156)
('european', 0.46917203068733215)
('airport', 0.4565390944480896)
('which', 0.4551011919975281)
('worldwide', 0.45195505023002625)
('countries', 0.44691070914268494)
('united', 0.44662800431251526)
('competition', 0.445771723985672)
('established', 0.4421483278274536)
('cooperation', 0.43993839621543884)
('national', 0.43693575263023376)
('well', 0.4362102746963501)
('efforts', 0.43539896607398987)
('organisation', 0.42948684096336365)
('geneva', 0.42751091718673706)
('its', 0.42686009407043457)
('community', 0.4250155985355377)
('the', 0.42279988527297974)
('foreign', 0.42203789949417114)
('media', 0.4203818142414093)


In [101]:
similar_words = word_vectors.most_similar('school', topn=25)
print('\n'.join([str(tup) for tup in similar_words]))

('schools', 0.7685322165489197)
('students', 0.7240082621574402)
('elementary', 0.719539999961853)
('college', 0.7110487222671509)
('education', 0.629763126373291)
('high', 0.6200570464134216)
('teacher', 0.6195809245109558)
('student', 0.6130647659301758)
('graduate', 0.6104811429977417)
('kindergarten', 0.5977890491485596)
('pupils', 0.580704927444458)
('teachers', 0.5805124044418335)
('graduating', 0.5789254307746887)
('university', 0.5767619013786316)
('secondary', 0.5760958790779114)
('graduated', 0.5723288655281067)
('teaching', 0.5692959427833557)
('attended', 0.5666410326957703)
('grades', 0.5653426647186279)
('enrolled', 0.5637966990470886)
('campus', 0.5635179877281189)
('taught', 0.5612472295761108)
('girls', 0.5602056980133057)
('classes', 0.5564578771591187)
('boys', 0.5541633367538452)


In [108]:
similar_words = word_vectors.most_similar(positive=['heidelberg','international','school'], topn=25)
print('\n'.join([str(tup) for tup in similar_words]))

('university', 0.6272861361503601)
('college', 0.5706648826599121)
('schools', 0.5535256266593933)
('students', 0.5433161854743958)
('academy', 0.535719096660614)
('universities', 0.533696711063385)
('graduate', 0.5333413481712341)
('established', 0.5204945802688599)
('student', 0.5086457133293152)
('graduating', 0.508064329624176)
('graduated', 0.5036836862564087)
('teaching', 0.49802365899086)
('studied', 0.49233201146125793)
('attended', 0.49102160334587097)
('faculty', 0.4878681004047394)
('taught', 0.4877416789531708)
('science', 0.48369601368904114)
('where', 0.48366373777389526)
('studies', 0.4816807806491852)
('education', 0.48023849725723267)
('at', 0.47949516773223877)
('cambridge', 0.4789472818374634)
('founded', 0.4779578447341919)
('in', 0.4772378206253052)
('berlin', 0.4744758903980255)


In [44]:
# See which words are closest to a given word in the vector space
similar_words = word_vectors.most_similar('germany', topn=25)
print('\n'.join([str(tup) for tup in similar_words]))

('german', 0.7490914463996887)
('austria', 0.6627424955368042)
('berlin', 0.6456044316291809)
('europe', 0.5969494581222534)
('munich', 0.5859135985374451)
('poland', 0.5792825222015381)
('switzerland', 0.5772391557693481)
('germans', 0.5746945738792419)
('denmark', 0.5592858195304871)
('france', 0.5567398071289062)
('netherlands', 0.5503618121147156)
('italy', 0.5471590757369995)
('frankfurt', 0.5468997359275818)
('belgium', 0.5435348153114319)
('sweden', 0.5273333191871643)
('britain', 0.5197895169258118)
('hungary', 0.517721951007843)
('cologne', 0.5129169821739197)
('stuttgart', 0.5022534132003784)
('nazi', 0.4997251331806183)
('austrian', 0.4918981194496155)
('european', 0.48993954062461853)
('hamburg', 0.4896203875541687)
('finland', 0.48852646350860596)
('schroeder', 0.4876728057861328)


In [45]:
# See which words are closest to a given word in the vector space
similar_words = word_vectors.most_similar('strange', topn=25)
print('\n'.join([str(tup) for tup in similar_words]))

('weird', 0.7194186449050903)
('bizarre', 0.6918294429779053)
('odd', 0.6896308660507202)
('peculiar', 0.64341139793396)
('mysterious', 0.6214640140533447)
('unusual', 0.5917140245437622)
('something', 0.5717740058898926)
('interesting', 0.5661023259162903)
('stranger', 0.5574067234992981)
('curious', 0.5372630953788757)
('things', 0.5328238010406494)
('sounds', 0.5287020206451416)
('surreal', 0.5256403088569641)
('scary', 0.5250455141067505)
('sort', 0.5225484371185303)
('frightening', 0.5155118107795715)
('wonderful', 0.515419602394104)
('fascinating', 0.5116831064224243)
('seems', 0.5087360739707947)
('thing', 0.5083068013191223)
('awful', 0.504219651222229)
('kind', 0.4996020793914795)
('coincidence', 0.4991011619567871)
('sorts', 0.49841076135635376)
('eerie', 0.49821218848228455)


### compute word relations

In [46]:
# Compute analogous words
# E.g. x is to queen as man is to king => x = v('queen') + v('man') - v('king')
analogous_words = word_vectors.most_similar(negative=['man'], positive=['king','woman'], topn=20)
print('\n'.join([str(tup) for tup in analogous_words]))

('queen', 0.6713276505470276)
('princess', 0.5432624220848083)
('throne', 0.5386104583740234)
('monarch', 0.5347574949264526)
('daughter', 0.49802514910697937)
('mother', 0.49564430117607117)
('elizabeth', 0.4832652509212494)
('kingdom', 0.47747090458869934)
('prince', 0.4668239951133728)
('wife', 0.46473273634910583)
('crown', 0.4479006826877594)
('her', 0.438046395778656)
('royal', 0.4296276569366455)
('marry', 0.42607834935188293)
('married', 0.42257994413375854)
('sister', 0.4174775779247284)
('anne', 0.41714465618133545)
('mary', 0.4165286123752594)
('duchess', 0.4059309959411621)
('margaret', 0.40583422780036926)


In [47]:
analogous_words = word_vectors.most_similar(negative=['apple'], positive=['grape','tree'], topn=20)
print('\n'.join([str(tup) for tup in analogous_words]))

('trees', 0.5595236420631409)
('vines', 0.5176296830177307)
('grapes', 0.4760022759437561)
('mistletoe', 0.45173028111457825)
('varieties', 0.4358935058116913)
('vine', 0.4358517825603485)
('planted', 0.43502077460289)
('shrubs', 0.4333231747150421)
('eucalyptus', 0.4284408688545227)
('flowering', 0.4223208427429199)
('plantings', 0.4216671288013458)
('shrub', 0.4141491651535034)
('merlot', 0.401363730430603)
('cane', 0.39994218945503235)
('oak', 0.39874598383903503)
('vineyard', 0.39725461602211)
('grapevines', 0.3902337849140167)
('syrah', 0.3878898620605469)
('fungus', 0.3868551552295685)
('varietals', 0.38517823815345764)


In [49]:
analogous_words = word_vectors.most_similar(negative=['apple'], positive=['grape','tree'], topn=20)
print('\n'.join([str(tup) for tup in analogous_words]))

('trees', 0.5595236420631409)
('vines', 0.5176296830177307)
('grapes', 0.4760022759437561)
('mistletoe', 0.45173028111457825)
('varieties', 0.4358935058116913)
('vine', 0.4358517825603485)
('planted', 0.43502077460289)
('shrubs', 0.4333231747150421)
('eucalyptus', 0.4284408688545227)
('flowering', 0.4223208427429199)
('plantings', 0.4216671288013458)
('shrub', 0.4141491651535034)
('merlot', 0.401363730430603)
('cane', 0.39994218945503235)
('oak', 0.39874598383903503)
('vineyard', 0.39725461602211)
('grapevines', 0.3902337849140167)
('syrah', 0.3878898620605469)
('fungus', 0.3868551552295685)
('varietals', 0.38517823815345764)


In [48]:
# Compute analogous words
# E.g. x is to queen as man is to king => x = v('queen') + v('man') - v('king')
analogous_words = word_vectors.most_similar(negative=['king'], positive=['queen','man'])
print('\n'.join([str(tup) for tup in analogous_words]))

('woman', 0.6957680583000183)
('girl', 0.5603842735290527)
('person', 0.5134302973747253)
('she', 0.4802548587322235)
('mother', 0.4633125066757202)
('boy', 0.46078377962112427)
('lady', 0.4552292823791504)
('teenager', 0.45107489824295044)
('her', 0.4438043236732483)
('men', 0.4426511228084564)


In [ ]:
# Compute analogous words
# E.g. x is to queen as man is to king => x = v('queen') + v('man') - v('king')
analogous_words = word_vectors.most_similar(negative=['mother'], positive=['father','daughter'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
# Compute analogous words
# E.g. x is to queen as man is to king => x = v('queen') + v('man') - v('king')
analogous_words = word_vectors.most_similar(negative=['mother'], positive=['father','girl'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
# Compute analogous words
# x is to daughter as people is to person (plural + daughter)
analogous_words = word_vectors.most_similar(negative=['person'], positive=['people','daughter'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
analogous_words = word_vectors.most_similar(negative=['soft'], positive=['hard','softer'])
# analogous_words = word_vectors.most_similar(negative=['soft'], positive=['hard','softest'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
analogous_words = word_vectors.most_similar(negative=['soft'], positive=['hard','softest'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
analogous_words = word_vectors.most_similar(negative=['doyle'], positive=['holmes','poirot'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
analogous_words = word_vectors.most_similar(negative=['sherlock'], positive=['holmes','poirot'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
analogous_words = word_vectors.most_similar(negative=['blue'], positive=['green', 'red'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
analogous_words = word_vectors.most_similar(negative=['bird'], positive=['sparrow', 'labrador']) # bulldog, chair, labrador
print('\n'.join([str(tup) for tup in analogous_words]))

In [50]:
analogous_words = word_vectors.most_similar(negative=['herd'], positive=['sheep', 'flock'])
print('\n'.join([str(tup) for tup in analogous_words]))

('flocks', 0.47732043266296387)
('pilgrims', 0.4477730095386505)
('flocked', 0.44382646679878235)
('goats', 0.4319022297859192)
('shepherds', 0.4304458498954773)
('chickens', 0.42081373929977417)
('geese', 0.40567657351493835)
('birds', 0.40210193395614624)
('thousands', 0.4013884365558624)
('cows', 0.38784632086753845)


In [53]:
analogous_words = word_vectors.most_similar(negative=['man'], positive=['woman', 'doctor'])
print('\n'.join([str(tup) for tup in analogous_words]))

('physician', 0.6098570823669434)
('nurse', 0.6059092879295349)
('doctors', 0.5913929343223572)
('pregnant', 0.5333701372146606)
('dentist', 0.5240339040756226)
('medical', 0.5112503170967102)
('pharmacist', 0.5043345093727112)
('surgeon', 0.5000936388969421)
('nurses', 0.4989488124847412)
('physicians', 0.49856632947921753)


In [59]:
analogous_words = word_vectors.most_similar(negative=['men'], positive=['husband', 'woman'])
print('\n'.join([str(tup) for tup in analogous_words]))

('wife', 0.7542847990989685)
('mother', 0.7439002990722656)
('daughter', 0.6886714696884155)
('grandmother', 0.6384336352348328)
('girlfriend', 0.6093586683273315)
('her', 0.591040849685669)
('herself', 0.58835768699646)
('niece', 0.5865375995635986)
('widow', 0.5809805989265442)
('granddaughter', 0.575857400894165)


Gender bias in word embeddings:
- men: doctor / woman: X
- men: pilot / woman: X
- men: programmer / woman: X
- men: ambitious / woman: X


In [84]:
analogous_words = word_vectors.most_similar(negative=['men'], positive=['capitan', 'woman'])
print('\n'.join([str(tup) for tup in analogous_words]))

('businesswoman', 0.3899881839752197)
('seductress', 0.3658837080001831)
('24-year-old', 0.3611169457435608)
('snaggletooth', 0.3501341938972473)
('priestess', 0.34973570704460144)
('basílica', 0.3448849022388458)
('picacho', 0.34088134765625)
('kafana', 0.3381454348564148)
('gridneva', 0.3349605202674866)
('re-rigged', 0.3349255919456482)


In [96]:
analogous_words = word_vectors.most_similar(negative=['woman'], positive=['actress', 'man'])
print('\n'.join([str(tup) for tup in analogous_words]))

('actor', 0.7867430448532104)
('starring', 0.6343069076538086)
('starred', 0.6015284061431885)
('movie', 0.5607398152351379)
('film', 0.5552588701248169)
('comedian', 0.5313838124275208)
('comedy', 0.5095089673995972)
('actors', 0.5039400458335876)
('star', 0.4965761601924896)
('stars', 0.4860335886478424)


In [95]:
analogous_words = word_vectors.most_similar(negative=['woman'], positive=['nurse', 'man'])
print('\n'.join([str(tup) for tup in analogous_words]))

('doctor', 0.4778822362422943)
('nurses', 0.4740598499774933)
('hospital', 0.4373469948768616)
('physician', 0.41594040393829346)
('sergeant', 0.40777313709259033)
('nursing', 0.39742162823677063)
('care', 0.3957793712615967)
('patient', 0.39143937826156616)
('paramedic', 0.3869837522506714)
('brother', 0.3712317645549774)


## Step 4A - optional. Using pre-trained models in Gensim

In many applications, you will simply want access to pre-trained word vectors (e.g. for plugging in to another model you are training). If you don't need the vectors to be tailored closely to your particular corpus, then you might like to use some pretrained models.

`word2vec` is not the only word embedding family of algorithms. Another, arguably even more powerful algorithm is the `FastText` algorithm, which Mikolov developed after moving to Facebook:

* Piotr Bojanowski and others, ‘Enriching Word Vectors with Subword Information’, ArXiv:1607.04606, 2017 <http://arxiv.org/abs/1607.04606>.

Instead of computing word vectors for each word, FastText splits each word into its constituent chunks. For example, 'cat' would be split into 'c', 'a', 't', 'ca', 'at' and 'cat', and 'burp' would be split into 'b', 'u', 'r', 'p', 'bu', 'ur', 'rp', 'bur', 'urp' and 'burp'. Then a vector is computer for each chunk that appears in the corpus. Each word is represented as the mean of all the chunks that make it up. FastText is able to learn very good word vectors because it can extract meaning from subword units, e.g. it can see that 'television', 'telegraph' and 'telepathy' all have 'tele' at the front, and can see that 'formality', 'criminality' and 'paucity' share subword units such as 'al' and 'ity'.

You can access many pretrained models using the Gensim downloader. Using the cells below, you can try out some of the different models available through Gensim. Along with `word2vec` and `FastText`, Gensim also supports `Glove` and `Doc2Vec` models.

**NB:** These trained models are very large, and will take a while to download. You may wish to download this notebook and execute the cells below on your own machine, in case Google kicks you out of the Colab environment.

In [ ]:
# See what models are on offer
print(list(api.info()['models'].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


#### word2vec-google-news-300

In [ ]:
# Optional! THIS WILL TAKE SOME TIME!!! (~15 min)
# 300-dimensional word vectors trained on a huge dataset from Google News
google_news_w2v = api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
analogous_words = google_news_w2v.most_similar(negative=['soft'], positive=['hard','softer'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
# x is to Kenya as Canberra is to Australia
google_news_w2v.most_similar(negative=['australia'],positive=['kenya','canberra'], topn=10)

#### glove_wiki_gigaword_300

In [ ]:
# download time ~ 4 min
glove_wiki_gigaword_300 = api.load('glove-wiki-gigaword-300')

[==================================================] 100.0% 376.1/376.1MB downloaded


In [ ]:
analogous_words = glove_wiki_gigaword_300.most_similar(negative=['soft'], positive=['hard','softer'])
print('\n'.join([str(tup) for tup in analogous_words]))

#### wikipedia_fasttext

In [ ]:
# Optional! THIS WILL TAKE SOME TIME!!! (download time ~10 minutes)
# Facebook's own FastText vectors, trained on Wikipedia
wikipedia_fasttext = api.load('fasttext-wiki-news-subwords-300')

[==================================================] 100.0% 958.5/958.4MB downloaded


In [ ]:
analogous_words = wikipedia_fasttext.most_similar(negative=['soft'], positive=['hard','softer'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
# x is to Wharton as London is to Dickens
analogous_words = wikipedia_fasttext.most_similar(negative=['paris'],positive=['france','london'], topn=10)
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
analogous_words = glove_wiki_gigaword_300.most_similar(negative=['soft'], positive=['hard','softer'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
# try your own examples...

## Step 5. Training models in other languages (Armenian, Georgian, Ukrainian ...)

Now let's train a model on our own corpus. You can try to create Armenian / Ukrainian / Russian / German / French word vector model.

Armenian, Ukrainian, Russian plaintext wikipedias are available at:
https://lindat.mff.cuni.cz/repository/xmlui/handle/11234/1-2735#

We can use the same size ~15 Million Words, (200MB) of these corpora to make sure the model is built within a reasonable time; however, when you have more time, you can try a more complete corpus.

It is possible to download/unzip/upload the files manually, however, it is faster to automatically load them directly from Heidelberg server space https://heibox.uni-heidelberg.de/d/42a07c9e95774e099a81/  into your colab environment. To do this run the cells 5a or 5b, or 5c, or 5d, or 5e -- depending on which language you prefer -- to download the data. Then skip other language and continue with Step5 -- building the model for your preferred language from the data you downloaded.


In [ ]:
!rm /usr/local/lib/python3.10/dist-packages/gensim/test/test_data/myOwnLangText8.txt
!rm index.html?dl=1

rm: cannot remove '/usr/local/lib/python3.10/dist-packages/gensim/test/test_data/myOwnLangText8.txt': No such file or directory
rm: cannot remove 'index.html?dl=1': No such file or directory


### 5c German - works!

In [ ]:
!wget https://heibox.uni-heidelberg.de/f/ca0a085347c34563a6e6/?dl=1

In [ ]:
!cp index.html?dl=1 /usr/local/lib/python3.10/dist-packages/gensim/test/test_data/myOwnLangText8.txt

In [ ]:
!head --lines=15 /usr/local/lib/python3.10/dist-packages/gensim/test/test_data/myOwnLangText8.txt

### Or: 5a Armenian

In [ ]:
!wget https://heibox.uni-heidelberg.de/f/ffe527ed3d1e4b4cb43a/?dl=1

In [ ]:
!cp index.html?dl=1 /usr/local/lib/python3.7/dist-packages/gensim/test/test_data/myOwnLangText8.txt

In [ ]:
!head --lines=15 /usr/local/lib/python3.7/dist-packages/gensim/test/test_data/myOwnLangText8.txt

### Or: 5b Ukrainian

In [ ]:
!wget https://heibox.uni-heidelberg.de/f/676bef3a6db8482e9665/?dl=1

In [ ]:
!cp index.html?dl=1 /usr/local/lib/python3.7/dist-packages/gensim/test/test_data/myOwnLangText8.txt

In [ ]:
!head --lines=15 /usr/local/lib/python3.7/dist-packages/gensim/test/test_data/myOwnLangText8.txt

### Or: 5d French

In [ ]:
!wget https://heibox.uni-heidelberg.de/f/5cdb485efd4046f2a457/?dl=1

In [ ]:
!cp index.html?dl=1 /usr/local/lib/python3.7/dist-packages/gensim/test/test_data/myOwnLangText8.txt

In [ ]:
!head --lines=15 /usr/local/lib/python3.7/dist-packages/gensim/test/test_data/myOwnLangText8.txt

### Or: 5f Georgian

In [ ]:
!wget https://heibox.uni-heidelberg.de/f/fa3509d869b949459f91/?dl=1

Lemmatized corpus

In [ ]:
!wget https://heibox.uni-heidelberg.de/f/b381c458ad5e4773a77d/?dl=1

In [ ]:
!cp index.html?dl=1 wiki_geo_lem.txt

In [ ]:
!head --lines=10 wiki_geo_lem.txt

In [ ]:
FIn = open("wiki_geo_lem.txt", 'r')
FOut = open('wiki_geo_l.txt', 'w')

i = 0
for SLine in FIn:
  i = i + 1
  SLine = SLine.strip() + ' '
  if i % 10000 == 0:
    SLine += '\n'

  FOut.write(SLine)

FOut.flush()
FOut.close()


In [ ]:
!head --lines=10 wiki_geo_l.txt

In [ ]:
!wc wiki_geo_lem.txt

 18054750  18054749 329202500 wiki_geo_lem.txt


In [ ]:
!cp index.html?dl=1 /usr/local/lib/python3.7/dist-packages/gensim/test/test_data/myOwnLangText8.txt

Alternatively, copy lemmatized corpus

In [ ]:
!cp wiki_geo_l.txt /usr/local/lib/python3.7/dist-packages/gensim/test/test_data/myOwnLangText8.txt

In [ ]:
!head --lines=15 /usr/local/lib/python3.7/dist-packages/gensim/test/test_data/myOwnLangText8.txt

### Stage6: Training own model

optional: to clean disk space, we remove the downloaded file

In [ ]:
# optional: to clean disk space, we remove the downloaded file
!rm index.html\?dl\=1

In [ ]:
from gensim.test.utils import datapath
from gensim import utils

class MyCorpus:
    """An iterator that yields sentences (lists of str)."""

    def __iter__(self):
        corpus_path = datapath('myOwnLangText8.txt')
        for line in open(corpus_path):
            # assume there's one document per line, tokens separated by whitespace
            yield utils.simple_preprocess(line)

In [ ]:
import gensim.models
corpusOwn = MyCorpus()

optional: examining what is in the corpus after standard preprocessing

In [ ]:
# Optional: Examining our corpus format
type(corpusOwn)

In [ ]:
dataOwn = [d for d in corpusOwn]

In [ ]:
print(len(dataOwn))
print(dataOwn[0])
print(len(dataOwn[0]))
print(dataOwn[4])
print(len(dataOwn[4]))
print(dataOwn[5])
print(len(dataOwn[5]))

... Initialising global parameters for our modelL vector size, collocation window, skip-grams, negative sampling....

In [ ]:
from gensim.models import Word2Vec # The word2vec model class
import gensim.downloader as api # Allows us to download some free training data

In [ ]:
my_vector_size = 100 # Dimensionality of the word vectors
window = 5 # How many words either side? (5 = 5 context words either side, i.e. 10 context words in total)
use_skip_gram = 1 # If you set this to 0, then it will create a 'continuous bag of words' model instead
use_softmax = 0 # If you set this to 1, then hierarchical softmax will be used instead of negative sampling
negative_samples = 5 # How many incorrect answers to generate per correct answer when negative sampling

modelOwn = Word2Vec(
    vector_size=my_vector_size,
    window=window,
    sg=use_skip_gram,
    hs=use_softmax,
    negative=negative_samples
)

... this cell may run for ~2 min or so...

In [ ]:
modelOwn.build_vocab(corpusOwn)

THIS MAY TAKE LONG!!! ... training the model may take 9 to 15 minutes... (just grab a cup of coffee or a sandwich while you are waiting... You may try chaning the number of epochs; if the number is lower the training is faster, but the quality may be lower...

In [ ]:
modelOwn.train(corpusOwn, epochs=5, total_examples=modelOwn.corpus_count)

(56967743, 75507725)

Now we copy word vectors and remove the model from memory (just to free up the resources...)

In [ ]:
# saving model
modelOwn.save("word2vecGerman5e.model")
!zip word2vecGerman5e.model.zip word2vecGerman5e.model word2vecGerman5e.model.syn1neg.npy word2vecGerman5e.model.wv.vectors.npy
# !rm word2vecGerman5e.model

In [ ]:
# extracting vectors from the model

In [ ]:
word_vectors_own = modelOwn.wv

In [ ]:
# del modelOwn

In [ ]:
word_vectors_own.save("word2vec.wordvectorsGerman5e")
!zip word2vec.wordvectorsGerman5e.zip word2vec.wordvectorsGerman5e word2vec.wordvectorsGerman5e.vectors.npy
# !rm word2vec.wordvectorsGerman5e

#### alternative: donloading pre-trained word vectors

In [ ]:
!wget https://heibox.uni-heidelberg.de/f/359e4cc37be8449fa142/?dl=1
!mv index.html?dl=1 word2vec.wordvectorsGerman5e.zip
!unzip word2vec.wordvectorsGerman5e.zip




In [ ]:
from gensim.models import KeyedVectors
word_vectors_own = KeyedVectors.load("word2vec.wordvectorsGerman5e", mmap='r')

Now we can examine the output of our word embeddings

### German examples

In [ ]:
# DE examples
# See the word vector for a particular word
vector = word_vectors_own['banane']
print(vector)

In [ ]:
# See which words are closest to a given word in the vector space
similar_words = word_vectors_own.most_similar('universität', topn=10)
print('\n'.join([str(tup) for tup in similar_words]))

In [ ]:
# See which words are closest to a given word in the vector space
similar_words = word_vectors_own.most_similar('rot', topn=10)
print('\n'.join([str(tup) for tup in similar_words]))

In [ ]:
# See which words are closest to a given word in the vector space
similar_words = word_vectors_own.most_similar('brüssel', topn=10)
print('\n'.join([str(tup) for tup in similar_words]))

In [ ]:
# See which words are closest to a given word in the vector space
similar_words = word_vectors_own.most_similar('deutschland', topn=10)
print('\n'.join([str(tup) for tup in similar_words]))

In [ ]:
# See which words are closest to a given word in the vector space
similar_words = word_vectors_own.most_similar('komisch', topn=10)
print('\n'.join([str(tup) for tup in similar_words]))

In [ ]:
# Compute analogous words
# E.g. x is to queen as man is to king => x = v('queen') + v('man') - v('king')
analogous_words = word_vectors_own.most_similar(negative=['mann'], positive=['könig','frau'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
# Compute analogous words
# E.g. x is to queen as man is to king => x = v('queen') + v('man') - v('king')
analogous_words = word_vectors_own.most_similar(negative=['könig'], positive=['königin','mann'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
# Compute analogous words
# E.g. x is to queen as man is to king => x = v('queen') + v('man') - v('king')
analogous_words = word_vectors_own.most_similar(negative=['mutter'], positive=['vater','tochter'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
# Compute analogous words
# E.g. x is to queen as man is to king => x = v('queen') + v('man') - v('king')
analogous_words = word_vectors_own.most_similar(negative=['mutter'], positive=['vater','mädchen'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
# Compute analogous words
# x is to daughter as people is to person (plural + daughter)
analogous_words = word_vectors_own.most_similar(negative=['person'], positive=['menschen','tochter'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
analogous_words = word_vectors_own.most_similar(negative=['weich'], positive=['hart','weicher'])
print('\n'.join([str(tup) for tup in analogous_words]))

### Georgian examples

In [ ]:
# See the word vector for a particular word -- 'world'
vector = word_vectors_own['სამყარო']
print(vector)

In [ ]:
# See which words are closest to a given word in the vector space = 'world'
similar_words = word_vectors_own.most_similar('არსება', topn=30)
print('\n'.join([str(tup) for tup in similar_words]))

In [ ]:
# blue
similar_words = word_vectors_own.most_similar('ლურჯი', topn=30)
print('\n'.join([str(tup) for tup in similar_words]))

In [ ]:
# france
similar_words = word_vectors_own.most_similar('საფრანგეთი', topn=10)
print('\n'.join([str(tup) for tup in similar_words]))

In [ ]:
# x is to king as woman is to man
analogous_words = word_vectors_own.most_similar(negative=['კაცი'], positive=['მეფე','ქალი'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
# x is to king as woman is to man
# x = small + biggest - big
# 𝑥−𝑣𝑒𝑐𝑡𝑜𝑟(′𝑠𝑚𝑎𝑙𝑙′)=𝑣𝑒𝑐𝑡𝑜𝑟(′𝑏𝑖𝑔𝑔𝑒𝑠𝑡′)−𝑣𝑒𝑐𝑡𝑜𝑟(′𝑏𝑖𝑔′)
# ∴ 𝑥=𝑣𝑒𝑐𝑡𝑜𝑟(′𝑠𝑚𝑎𝑙𝑙′)+𝑣𝑒𝑐𝑡𝑜𝑟(′𝑏𝑖𝑔𝑔𝑒𝑠𝑡′)−𝑣𝑒𝑐𝑡𝑜𝑟(′𝑏𝑖𝑔′)
# ∴ 𝑥=𝑣𝑒𝑐𝑡𝑜𝑟(′փոքր′)+𝑣𝑒𝑐𝑡𝑜𝑟(′ամենամեծը′)−𝑣𝑒𝑐𝑡𝑜𝑟(′մեծ′)
analogous_words = word_vectors_own.most_similar(negative=['პირი'], positive=['ხალხი','ქალიშვილი'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
# try your own examples... (also -- morphology?)
# # x is to daughter as people is to person (plural + daughter)
# analogous_words = word_vectors.most_similar(negative=['person'], positive=['people','daughter'])

### Armenian examples

In [ ]:
# See the word vector for a particular word -- 'world'
vector = word_vectors_own['աշխարհը']
print(vector)

In [ ]:
# See which words are closest to a given word in the vector space = 'world'
similar_words = word_vectors_own.most_similar('աշխարհը', topn=10)
print('\n'.join([str(tup) for tup in similar_words]))

In [ ]:
# blue
similar_words = word_vectors_own.most_similar('կապույտ', topn=10)
print('\n'.join([str(tup) for tup in similar_words]))

In [ ]:
# france
similar_words = word_vectors_own.most_similar('ֆրանսիա', topn=10)
print('\n'.join([str(tup) for tup in similar_words]))

In [ ]:
# x is to king as woman is to man
analogous_words = word_vectors_own.most_similar(negative=['մարդ'], positive=['թագավոր','կին'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
# x is to king as woman is to man
# x = small + biggest - big
# 𝑥−𝑣𝑒𝑐𝑡𝑜𝑟(′𝑠𝑚𝑎𝑙𝑙′)=𝑣𝑒𝑐𝑡𝑜𝑟(′𝑏𝑖𝑔𝑔𝑒𝑠𝑡′)−𝑣𝑒𝑐𝑡𝑜𝑟(′𝑏𝑖𝑔′)
# ∴ 𝑥=𝑣𝑒𝑐𝑡𝑜𝑟(′𝑠𝑚𝑎𝑙𝑙′)+𝑣𝑒𝑐𝑡𝑜𝑟(′𝑏𝑖𝑔𝑔𝑒𝑠𝑡′)−𝑣𝑒𝑐𝑡𝑜𝑟(′𝑏𝑖𝑔′)
# ∴ 𝑥=𝑣𝑒𝑐𝑡𝑜𝑟(′փոքր′)+𝑣𝑒𝑐𝑡𝑜𝑟(′ամենամեծը′)−𝑣𝑒𝑐𝑡𝑜𝑟(′մեծ′)
analogous_words = word_vectors_own.most_similar(negative=['մեծ'], positive=['ամենամեծը','փոքր'])
print('\n'.join([str(tup) for tup in analogous_words]))

In [ ]:
# try your own examples... (also -- morphology?)
# # x is to daughter as people is to person (plural + daughter)
# analogous_words = word_vectors.most_similar(negative=['person'], positive=['people','daughter'])